# Lab: apparent resistivity from electric and magnetic time series

```{note}
This lab accompanies {doc}`magnetotellurics`. The chapter described the impedance and the skin effect; this lab builds them from scratch, starting with a pair of time series and ending with a resistivity-versus-depth picture.
```

## Where the data lives — and why this lab uses synthetic time series

Magnetotelluric data comes from two places. Processed transfer functions — impedances already estimated from raw recordings — are archived in the EMTF/SPUD database at IRIS/EarthScope (https://ds.iris.edu/spud/emtf), which holds thousands of stations worldwide and can be queried by region, period range, and quality. That is the archive to reach for if you want real impedance tensors. Raw electric and magnetic time series are a different story. A handful of experiments have deposited raw recordings through IRIS (and now EarthScope's data services), but the coverage is sparse compared with seismic networks, because magnetotelluric instruments are not installed permanently and the logistics of polar fieldwork have kept Antarctic datasets scarce.

This lab therefore runs on a synthetic-but-realistic time series generated below. The physics is the same; the numbers come from a plausible three-layer model rather than a specific field experiment. Two stubs in the first code cell show how to swap in real data where it exists.

**Data volume.** A typical field MT station samples electric and magnetic fields at 1–512 Hz on four channels. At 1 Hz, a month of continuous recording produces roughly 10 MB per channel, so a full station-month at broadband rates runs to a few hundred MB and at high rates to a few GB. That is modest by geophysical standards. The polar challenge is not storage but logistics: deploying non-polarizing electrodes that maintain good electrical contact with snow-covered ice over weeks, keeping electronics warm enough to function, and retrieving the instruments before a winter shutdown. Electrode contact resistance is the dominant noise source in polar MT, not the volume of data.

```{admonition} Patterns from UW-GDA
:class: seealso
The windowed FFT and cross-spectral processing at the core of this lab sits squarely in the domain of the UW Geospatial Data Analysis course of {cite}`shean2023`, which teaches NumPy array operations and spectral methods on geophysical time series. Module 3 (NumPy, Pandas, and Matplotlib) covers the FFT and windowing fundamentals the impedance estimator here relies on, and Module 8 (vector time series and time-series handling) addresses the timestamp alignment and multi-channel bookkeeping that carry over directly to multi-component MT recordings. Work through those two modules there if the tooling here feels unfamiliar.
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq

# ── global constants ──────────────────────────────────────────────────────────
MU0 = 4.0 * np.pi * 1e-7      # free-space permeability, H/m

# ── three-layer model: ice / till / rock ──────────────────────────────────────
# Each layer is (resistivity in Ohm·m, thickness in m); the last layer is a
# halfspace so its thickness is ignored.
MODEL = [
    (1e6, 1000.0),   # ice: 1 MOhm·m, 1 km thick
    (1.0,  200.0),   # water-saturated till: 1 Ohm·m, 200 m thick
    (300.0,  None),  # crystalline basement halfspace: 300 Ohm·m
]

# ── time-series parameters ────────────────────────────────────────────────────
DT = 1.0          # sample interval, s  (1 Hz)
N_TOTAL = 2**18   # total samples (~3 days at 1 Hz); 2^18 for fast FFT sizing
SEED = 42

print(f"Sample rate : {1/DT:.1f} Hz")
print(f"Record length: {N_TOTAL * DT / 86400:.1f} days")

# ── # verify: real data stub (IRIS FDSN web services) ─────────────────────────
# To replace synthetic data with a real station, uncomment and fill in:
#
# from obspy.clients.fdsn import Client
# client = Client('IRIS')
# st = client.get_waveforms(
#     network='EM', station='WHI01', location='*',
#     channel='LF*',   # LFE, LFN = electric; LFX, LFY = magnetic
#     starttime=UTCDateTime('2018-01-01'),
#     endtime=UTCDateTime('2018-01-08')
# )
# For processed transfer functions from EMTF/SPUD:
# https://ds.iris.edu/spud/emtf  → download as .xml or .edi
print("Real-data stubs are in the comments above.")

## Computing the theoretical impedance for a layered halfspace

For a stack of horizontal layers with resistivities $\rho_1, \rho_2, \ldots$ and thicknesses $h_1, h_2, \ldots$ above a halfspace, the impedance at the surface is computed recursively from the bottom up. Starting at the halfspace, the impedance is the half-space intrinsic impedance,

$$Z_n(\omega) = \sqrt{i\omega\mu_0\rho_n},$$

and working upward through each layer the recursion is

$$Z_k(\omega) = \sqrt{i\omega\mu_0\rho_k} \;\frac{Z_{k+1} + \sqrt{i\omega\mu_0\rho_k}\tanh(k_k h_k)}{\sqrt{i\omega\mu_0\rho_k} + Z_{k+1}\tanh(k_k h_k)}, \qquad k_k = \sqrt{\frac{i\omega\mu_0}{\rho_k}},$$

where the branch cut for the square root is chosen so that $\mathrm{Im}(k_k) > 0$ (decaying wave). This is the propagator-matrix method in its simplest form [TODO-CITE: wait1954 or tikhonov1950]. The apparent resistivity and phase at each frequency follow directly from $Z(\omega)$:

$$\rho_a(\omega) = \frac{|Z(\omega)|^2}{\omega\mu_0}, \qquad \phi(\omega) = \arg Z(\omega).$$

For a uniform halfspace $\rho_a = \rho$ at all frequencies and $\phi = 45°$. Layering curves $\rho_a$ away from those baselines, and reading those curves in terms of the layer model is the inversion problem.

In [ ]:
def wavenumber(omega, rho):
    """Vertical wavenumber for a layer: k = sqrt(i*omega*mu0/rho).

    Chooses the root with Im(k) > 0 (downward-decaying field).
    """
    k = np.sqrt(1j * omega * MU0 / rho)
    # ensure Im(k) > 0
    k = np.where(k.imag < 0, -k, k)
    return k


def halfspace_Z(omega, rho):
    """Intrinsic impedance of a uniform halfspace."""
    return np.sqrt(1j * omega * MU0 * rho)


def layered_Z(omega, model):
    """Surface impedance for a stack of layers by downward recursion.

    model : list of (rho, thickness) tuples; last entry is the halfspace
            and its thickness is ignored.
    omega : angular frequency array (rad/s), shape (nf,)
    Returns Z with shape (nf,).
    """
    rho_hs, _ = model[-1]
    Z = halfspace_Z(omega, rho_hs)
    for rho, h in reversed(model[:-1]):
        k = wavenumber(omega, rho)
        Z_int = np.sqrt(1j * omega * MU0 * rho)
        th = np.tanh(k * h)
        Z = Z_int * (Z + Z_int * th) / (Z_int + Z * th)
    return Z


def apparent_resistivity(Z, omega):
    """Apparent resistivity (Ohm·m) and phase (degrees) from Z and omega."""
    rho_a = np.abs(Z)**2 / (omega * MU0)
    phase = np.degrees(np.angle(Z))
    return rho_a, phase


# Compute the true curve for our three-layer model
periods_true = np.logspace(-1, 6, 300)   # 0.1 s to 10^6 s (~11.6 days)
omega_true = 2.0 * np.pi / periods_true
Z_true = layered_Z(omega_true, MODEL)
rho_a_true, phase_true = apparent_resistivity(Z_true, omega_true)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax1.loglog(periods_true, rho_a_true, 'k', lw=2, label='three-layer model')
ax1.set_ylabel(r'$\rho_a$ ($\Omega$·m)')
ax1.set_ylim(0.1, 3e6)
ax1.legend()
ax2.semilogx(periods_true, phase_true, 'k', lw=2)
ax2.axhline(45.0, color='gray', ls=':', lw=1, label='45° halfspace')
ax2.set_ylabel('phase (°)')
ax2.set_xlabel('period (s)')
ax2.set_ylim(0, 90)
ax2.legend()
fig.suptitle('Theoretical apparent resistivity — ice / till / rock model')
plt.tight_layout()
plt.show()

## Generating synthetic E and B time series

A real MT recording measures the horizontal electric field $E_x(t)$ and the perpendicular horizontal magnetic field $B_y(t)$ (and their orthogonal pair), with the relationship $E_x(\omega) = Z(\omega) B_y(\omega) / \mu_0$ in the frequency domain. To make realistic synthetic recordings, the procedure is:

1. Generate a broadband magnetic field $B_y(t)$ with a natural MT spectrum. The geomagnetic source field is redder than white noise, roughly falling as $1/f^{0.5}$ from the micropulsation band through the audio-magnetotelluric range [TODO-CITE: vozoff1972].
2. Apply the theoretical impedance of the three-layer model in the frequency domain to produce the corresponding electric field $E_x(\omega)$.
3. Transform back to time, then add independent Gaussian noise to both $E_x$ and $B_y$ to mimic electrode drift and instrument noise.

The noise on the electric channel is typically larger relative to signal than on the magnetic channel, because electrode contact resistance with ice is high and variable. That asymmetry is the main motivation for the remote-reference technique that Task 6 asks you to think about.

In [ ]:
rng = np.random.default_rng(SEED)

# ── 1. broadband magnetic source in the frequency domain ─────────────────────
freqs = rfftfreq(N_TOTAL, d=DT)            # Hz;  shape (N_TOTAL//2 + 1,)
freqs_safe = np.where(freqs == 0, np.inf, freqs)
omega = 2.0 * np.pi * freqs_safe           # rad/s

# Random phase, amplitude ~ f^{-0.5} (natural MT source spectrum)
phase_rand = rng.uniform(0.0, 2.0 * np.pi, size=freqs.size)
amplitude_B = freqs_safe**(-0.5)           # spectral shape; arbitrary units → nT
amplitude_B[0] = 0.0                       # zero DC
B_y_fft = amplitude_B * np.exp(1j * phase_rand)

# ── 2. electric field from the impedance ─────────────────────────────────────
Z_freq = layered_Z(omega, MODEL)           # complex impedance at each freq
Z_freq[0] = 0.0                            # kill DC
# E_x = Z * B_y / mu0   (SI: E in V/m, B in T)
E_x_fft = Z_freq / MU0 * B_y_fft

# ── 3. transform to time and add noise ────────────────────────────────────────
from numpy.fft import irfft

B_y_clean = irfft(B_y_fft, n=N_TOTAL)     # nT-scale (arbitrary normalisation)
E_x_clean = irfft(E_x_fft, n=N_TOTAL)     # mV/km-scale

# noise levels: 5% rms on B, 20% rms on E (high electrode-noise scenario)
noise_B = 0.05 * B_y_clean.std() * rng.standard_normal(N_TOTAL)
noise_E = 0.20 * E_x_clean.std() * rng.standard_normal(N_TOTAL)

B_y = B_y_clean + noise_B
E_x = E_x_clean + noise_E

t_ts = np.arange(N_TOTAL) * DT / 3600.0   # time in hours for plotting

fig, (axB, axE) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
win = slice(0, 6 * 3600)                   # first 6 hours
axB.plot(t_ts[win], B_y[win], lw=0.5)
axB.set_ylabel(r'$B_y$ (a.u.)')
axE.plot(t_ts[win], E_x[win], lw=0.5, color='C1')
axE.set_ylabel(r'$E_x$ (a.u.)')
axE.set_xlabel('time (hours)')
fig.suptitle('First 6 hours of synthetic MT time series')
plt.tight_layout()
plt.show()

## Processing: windowed FFT, cross-spectra, and impedance estimation

Estimating the impedance from a pair of time series is a spectral problem. The relationship $E_x(\omega) = Z(\omega)\,H_y(\omega)$ holds frequency by frequency, where $H_y = B_y/\mu_0$ is the magnetic field intensity. With noisy data the best estimate is not just the ratio of two Fourier coefficients at each frequency; instead, the standard approach averages cross-spectral estimates over many data windows and forms

$$\hat{Z}(\omega) = \frac{\langle E_x(\omega)\,H_y^*(\omega)\rangle}{\langle H_y(\omega)\,H_y^*(\omega)\rangle},$$

which is an ordinary least-squares estimate of the complex scalar $Z$ at each frequency bin. The angle brackets denote averaging over windows. Each window is Hann-tapered before the FFT to reduce spectral leakage, and after estimation the frequency axis is decimated onto a logarithmically spaced grid so that periods per decade are uniform at all scale lengths — this is called frequency-band averaging or geometric-mean stacking [TODO-CITE: egbert1997].

The coherence between $E_x$ and $H_y$ in each band is a quality indicator. High coherence (close to 1) means the fields are well-related and the impedance estimate is reliable; low coherence flags noisy or incoherent windows that should be downweighted or discarded.

In [ ]:
def hann(n):
    """Hann window of length n."""
    return 0.5 * (1.0 - np.cos(2.0 * np.pi * np.arange(n) / (n - 1)))


def estimate_impedance(E, B, dt, window_len, overlap=0.5):
    """Estimate Z(f) = <E·H*> / <H·H*> by windowed cross-spectral averaging.

    E, B        : time series arrays (same length)
    dt          : sample interval, s
    window_len  : samples per window (should be a power of 2)
    overlap     : fractional overlap between consecutive windows (0–1)

    Returns freqs (Hz), Z (complex), coherence (real, 0–1).
    """
    step = int(window_len * (1.0 - overlap))
    win = hann(window_len)
    freqs = rfftfreq(window_len, d=dt)

    # accumulators for cross-spectra
    SEH = np.zeros(freqs.size, dtype=complex)
    SHH = np.zeros(freqs.size, dtype=complex)
    SEE = np.zeros(freqs.size, dtype=complex)
    n_wins = 0

    start = 0
    H = B / MU0
    while start + window_len <= len(E):
        e_seg = E[start:start + window_len] * win
        h_seg = H[start:start + window_len] * win
        E_fft = rfft(e_seg)
        H_fft = rfft(h_seg)
        SEH += E_fft * H_fft.conj()
        SHH += H_fft * H_fft.conj()
        SEE += E_fft * E_fft.conj()
        n_wins += 1
        start += step

    Z_est = SEH / SHH
    coherence = np.abs(SEH)**2 / (np.abs(SEE) * np.abs(SHH))
    return freqs, Z_est, np.sqrt(coherence.real), n_wins


WINDOW_LEN = 2**12    # 4096 samples = ~68 min at 1 Hz
freqs_est, Z_est, coh_est, nw = estimate_impedance(
    E_x, B_y, DT, WINDOW_LEN, overlap=0.75
)

print(f"Windows used: {nw}")
print(f"Frequency resolution: {freqs_est[1]:.4f} Hz")

# restrict to sensible frequency range (avoid DC and Nyquist ends)
mask = (freqs_est > 1e-4) & (freqs_est < 0.4)
f_plot = freqs_est[mask]
T_plot = 1.0 / f_plot
Z_plot = Z_est[mask]
coh_plot = coh_est[mask]

omega_plot = 2.0 * np.pi * f_plot
rho_a_est, phase_est = apparent_resistivity(Z_plot, omega_plot)

fig, ax = plt.subplots(figsize=(8, 3))
sc = ax.scatter(T_plot, rho_a_est, c=coh_plot, s=4, cmap='viridis',
                vmin=0.5, vmax=1.0, zorder=3)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('period (s)')
ax.set_ylabel(r'$\rho_a$ ($\Omega$·m)')
ax.set_title('Raw impedance estimates, colored by coherence')
plt.colorbar(sc, ax=ax, label='coherence')
plt.tight_layout()
plt.show()

## Band-averaging onto a logarithmic period grid

The raw cross-spectral estimates at each DFT bin are noisy. The standard practice is to average them in geometric-mean frequency bands — roughly a fixed number of bins per octave — so that the resulting curve has uniform point density on a log-period axis and each estimate has been averaged over enough independent windows to suppress scatter. The coherence within each band is a natural weight: bins with low coherence contribute less to the band average.

In [ ]:
def band_average(periods, Z, coherence, n_per_decade=8):
    """Average impedance estimates into logarithmically spaced period bands.

    Returns band-center periods, averaged Z, averaged coherence.
    """
    log_p = np.log10(periods)
    p_min, p_max = log_p.min(), log_p.max()
    n_bands = int(np.round((p_max - p_min) * n_per_decade))
    edges = np.linspace(p_min, p_max, n_bands + 1)

    p_out, Z_out, coh_out = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        idx = (log_p >= lo) & (log_p < hi)
        if idx.sum() < 2:
            continue
        w = coherence[idx]                      # coherence as weight
        w = w / w.sum()
        Z_mean = (w * Z[idx]).sum()             # weighted mean of complex Z
        p_out.append(10.0**((lo + hi) / 2.0))
        Z_out.append(Z_mean)
        coh_out.append(coherence[idx].mean())

    return (np.array(p_out), np.array(Z_out, dtype=complex),
            np.array(coh_out))


T_band, Z_band, coh_band = band_average(T_plot, Z_plot, coh_plot,
                                         n_per_decade=10)
omega_band = 2.0 * np.pi / T_band
rho_a_band, phase_band = apparent_resistivity(Z_band, omega_band)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

ax1.loglog(periods_true, rho_a_true, 'k', lw=2, label='true model')
ax1.errorbar(T_band, rho_a_band, fmt='o', ms=5, color='C0',
             label='estimated (band-averaged)')
ax1.set_ylabel(r'$\rho_a$ ($\Omega$·m)')
ax1.set_ylim(0.1, 3e6)
ax1.legend()

ax2.semilogx(periods_true, phase_true, 'k', lw=2, label='true model')
ax2.plot(T_band, phase_band, 'o', ms=5, color='C0',
         label='estimated (band-averaged)')
ax2.axhline(45.0, color='gray', ls=':', lw=1)
ax2.set_ylabel('phase (°)')
ax2.set_xlabel('period (s)')
ax2.set_ylim(0, 90)
ax2.legend()

fig.suptitle('Estimated vs. true apparent resistivity and phase')
plt.tight_layout()
plt.show()

## The ice problem: why resistive ice shifts the sensitivity window

The skin depth at angular frequency $\omega$ in a medium of resistivity $\rho$ is

$$\delta = \sqrt{\frac{2\rho}{\omega\mu_0}} = 503\sqrt{\frac{\rho}{f}} \text{ m},$$

where $f$ is frequency in Hz and $\rho$ is in $\Omega\cdot\mathrm{m}$. For ice at $10^6\,\Omega\cdot\mathrm{m}$, a 1-Hz signal has a skin depth of about 500 km — the entire ice sheet is electromagnetically transparent at that frequency. The ice contributes essentially no signal to the impedance at short periods. But it does something more subtle at long periods: it shifts the apparent resistivity curve upward at all periods shorter than those that see through the till into the basement.

The practical consequence is that to resolve a conductive till layer beneath 1 km of ice, you need periods long enough that the skin depth in the till itself — which at $1\,\Omega\cdot\mathrm{m}$ is only $\sim 500$ m at a 1-s period — can penetrate to depth. Because the ice does not attenuate the signal, the "period budget" for a polar MT survey is set by the till and basement resistivities, not the ice thickness. The ice is invisible but it delays the interpreter: the apparent-resistivity curve at short periods simply reads $10^6\,\Omega\cdot\mathrm{m}$, giving no layer information, and the useful part of the sounding only begins at periods of tens to hundreds of seconds.

In [ ]:
def skin_depth(rho, period):
    """Skin depth in metres: delta = sqrt(2*rho*T / (2*pi*mu0))."""
    omega = 2.0 * np.pi / period
    return np.sqrt(2.0 * rho / (omega * MU0))


T_sd = np.logspace(-1, 6, 300)
rho_ice = 1e6
rho_till = 1.0
rho_rock = 300.0

# Compare two models: (a) original ice/till/rock, (b) replace ice with rock
model_no_ice = [
    (rho_rock, 1000.0),   # rock at the surface (no ice)
    (rho_till,  200.0),
    (rho_rock,  None),
]

omega_sd = 2.0 * np.pi / T_sd
rho_a_no_ice, _ = apparent_resistivity(layered_Z(omega_sd, model_no_ice),
                                        omega_sd)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.loglog(T_sd, skin_depth(rho_ice, T_sd), label=r'ice (10$^6$ $\Omega$·m)')
ax.loglog(T_sd, skin_depth(rho_till, T_sd), label=r'till (1 $\Omega$·m)')
ax.loglog(T_sd, skin_depth(rho_rock, T_sd), label=r'rock (300 $\Omega$·m)')
ax.axhline(1000.0, color='gray', ls='--', lw=1, label='ice thickness')
ax.axhline(1200.0, color='brown', ls='--', lw=1, label='ice + till base')
ax.set_xlabel('period (s)')
ax.set_ylabel('skin depth (m)')
ax.set_title('Skin depth by lithology')
ax.legend(fontsize=8)

ax = axes[1]
ax.loglog(T_sd, rho_a_true[:len(T_sd)], 'k', lw=2, label='ice / till / rock')
ax.loglog(T_sd, rho_a_no_ice, 'C3--', lw=2, label='rock / till / rock (no ice)')
ax.set_xlabel('period (s)')
ax.set_ylabel(r'$\rho_a$ ($\Omega$·m)')
ax.set_title('Effect of ice on the sounding curve')
ax.legend()

plt.tight_layout()
plt.show()

# Print the period at which skin depth in the till first exceeds 200 m (the till base)
till_200 = T_sd[skin_depth(rho_till, T_sd) >= 200.0][0]
till_1200 = T_sd[skin_depth(rho_till, T_sd) >= 1200.0][0]
print(f"Till skin depth reaches till-base (200 m) at T = {till_200:.1f} s")
print(f"Till skin depth reaches rock (1200 m) at T = {till_1200:.1f} s")
print(f"Ice skin depth at 1 s: {skin_depth(rho_ice, 1.0)/1e3:.0f} km (transparent)")

## Tasks

**Task 1 — Noise sensitivity.**
The synthetic time series above was generated with 20% electric-channel noise. Re-run the estimation pipeline with noise levels of 5%, 50%, and 100% on the electric channel (keeping the magnetic-channel noise at 5%). For each case, plot the estimated $\rho_a(T)$ against the true curve and note the period range over which the estimate degrades. At what noise level does the till layer become invisible in the estimated curve?

**Task 2 — Window length and period resolution.**
The window length `WINDOW_LEN` controls both the longest period resolvable (one period per window) and the number of independent averages available at shorter periods. Re-run the estimation with window lengths of $2^{10}$, $2^{12}$, and $2^{14}$ samples. Plot the resulting $\rho_a$ curves and explain the trade-off in terms of variance and bias at long periods.

**Task 3 — Remote reference.**
A known weakness of the single-station impedance estimator above is that correlated noise on both E and B channels biases the cross-spectral estimate. In the remote-reference method, a second magnetometer is deployed far from the local station (far enough that its noise is uncorrelated with the local noise), and $H_y^*$ in the numerator and denominator above is replaced by $H_y^{\mathrm{ref}*}$ from the remote site.

In this task you do not need to simulate a real remote reference. Instead, describe in a few sentences (a) why the remote-reference substitution eliminates the bias from local magnetic noise, and (b) why it does *not* eliminate the bias from local electric-channel noise. Then sketch (in words or pseudocode) how you would modify `estimate_impedance` to accept an optional remote-reference magnetic time series.

In [ ]:
# Task 1: noise sensitivity
# Modify the noise_E level and re-run estimate_impedance + band_average.
# Suggested structure:

noise_levels = [0.05, 0.20, 0.50, 1.00]   # fraction of E rms
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

for noise_frac in noise_levels:
    noise_E_task = noise_frac * E_x_clean.std() * rng.standard_normal(N_TOTAL)
    E_noisy = E_x_clean + noise_E_task
    # YOUR CODE HERE:
    # f_e, Z_e, coh_e, _ = estimate_impedance(E_noisy, B_y, DT, WINDOW_LEN, 0.75)
    # mask_e = (f_e > 1e-4) & (f_e < 0.4)
    # T_e, Z_e2, coh_e2 = band_average(1.0 / f_e[mask_e], Z_e[mask_e], coh_e[mask_e])
    # rho_e, phase_e = apparent_resistivity(Z_e2, 2*np.pi / T_e)
    # ax1.loglog(T_e, rho_e, label=f'{noise_frac*100:.0f}% E noise')
    # ax2.semilogx(T_e, phase_e, label=f'{noise_frac*100:.0f}% E noise')
    pass

ax1.loglog(periods_true, rho_a_true, 'k--', lw=2, label='true')
ax1.set_ylabel(r'$\rho_a$ ($\Omega$·m)')
ax1.set_ylim(0.1, 3e6)
ax1.legend(fontsize=8)
ax2.semilogx(periods_true, phase_true, 'k--', lw=2)
ax2.set_ylabel('phase (°)')
ax2.set_xlabel('period (s)')
plt.tight_layout()
plt.show()

## What a conductive anomaly under an ice stream means

The {doc}`magnetotellurics` chapter described the Gustafson et al. survey across Whillans Ice Stream, which found a deep conductive layer interpreted as a large body of saline groundwater in the sediments below the ice {cite}`gustafson2022`. That result reshapes the picture that {doc}`../thermomechanics/hydrology` builds of the subglacial water budget. The thin film of meltwater at the bed — the focus of most drainage models — sits above a far larger reservoir that exchanges with it on timescales that are not yet fully constrained.

Three classes of anomaly are distinguishable in an MT sounding beneath ice.

A moderately conductive layer ($\sim 1\text{–}10\,\Omega\cdot\mathrm{m}$) at till depth is consistent with water-saturated unconsolidated sediment — the deforming till that the sliding laws of {doc}`../thermomechanics/basal-motion` treat as a viscous layer. Till conductivity scales with porosity and pore-fluid salinity; a saltier, more porous till is softer, drains differently, and produces different basal drag.

A very conductive body ($< 0.1\,\Omega\cdot\mathrm{m}$) at depth, with gradients sharper than a diffuse till, is more consistent with a brine or saline-groundwater reservoir in fractured rock or unconsolidated sediment. These bodies can be ancient, isolated from the modern hydrological cycle, or actively connected to the basal system.

A transition from conductive to resistive with depth traces the boundary between thawed and frozen sediment, which is a direct map of the geothermal heat budget that {doc}`../thermomechanics/basal-motion` and {doc}`../thermomechanics/hydrology` need as a lower boundary condition.

What ties these observations to dynamics is the ice-stream surface. Changes in basal water pressure leave a signal in elevation through the coupling described in {doc}`elevation`: a till that transitions from drained to saturated lowers basal drag, accelerates the ice, and thins the column — a sequence that a repeat altimetry mission can detect from above. Magnetotellurics provides the resistivity map; {doc}`elevation` provides the response; and basal mechanics provides the model that connects them. The observation threads are underused in combination, which is one reason the subglacial water budget remains uncertain even for well-studied ice streams like Whillans.

In [ ]:
# Compare three interpretations of a conductive anomaly under 1 km of ice.

models_interp = {
    'deforming till (10 Ohm·m)': [
        (1e6, 1000.0),
        (10.0,  200.0),
        (300.0,  None),
    ],
    'saline brine (0.05 Ohm·m)': [
        (1e6, 1000.0),
        (0.05,  100.0),
        (300.0,  None),
    ],
    'thaw boundary (gradual)': [
        (1e6, 1000.0),
        (50.0,   500.0),   # partially frozen sediment
        (1.0,    300.0),   # thawed sediment
        (300.0,   None),
    ],
    'original (1 Ohm·m till)': MODEL,
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
T_cmp = np.logspace(0, 6, 400)
omega_cmp = 2.0 * np.pi / T_cmp

for label, mdl in models_interp.items():
    Z_cmp = layered_Z(omega_cmp, mdl)
    rho_cmp, phi_cmp = apparent_resistivity(Z_cmp, omega_cmp)
    ax1.loglog(T_cmp, rho_cmp, lw=2, label=label)
    ax2.semilogx(T_cmp, phi_cmp, lw=2, label=label)

ax1.set_ylabel(r'$\rho_a$ ($\Omega$·m)')
ax1.legend(fontsize=8)
ax2.set_ylabel('phase (°)')
ax2.set_xlabel('period (s)')
ax2.axhline(45.0, color='gray', ls=':', lw=1)
fig.suptitle('Apparent resistivity for three interpretations of a subglacial anomaly')
plt.tight_layout()
plt.show()

# At what period (in seconds and in days) does each model first deviate by
# more than 10% from the ice-only halfspace?
rho_a_ice_only, _ = apparent_resistivity(halfspace_Z(omega_cmp, 1e6), omega_cmp)
print("Period at which each model first deviates >10% from ice-only:")
for label, mdl in models_interp.items():
    Z_cmp = layered_Z(omega_cmp, mdl)
    rho_cmp, _ = apparent_resistivity(Z_cmp, omega_cmp)
    dev = np.abs(rho_cmp - rho_a_ice_only) / rho_a_ice_only
    idx = np.argmax(dev > 0.10)
    if dev[idx] > 0.10:
        print(f"  {label:40s}  T = {T_cmp[idx]:.1f} s  ({T_cmp[idx]/86400:.2f} days)")
    else:
        print(f"  {label:40s}  never (< 10% deviation in range)")

## Synthesis

Work through the following questions in a short paragraph or bullet points, drawing on the curves you have produced.

**1. Reading the layers back out.** For the original three-layer model, identify in your band-averaged $\rho_a(T)$ curve the short-period plateau (sampling ice), the transition to the conductive till, and the rise back toward the basement resistivity at long periods. At what periods do the transitions occur, and do they match the skin-depth estimates from the ice-problem cell?

**2. The data-coverage gap.** This lab generated 3 days of 1-Hz data. The longest period resolvable is one window length. A real long-period MT sounding needs periods of thousands to tens of thousands of seconds to see beneath thick Antarctic ice and into the crust. Calculate how many days of continuous recording you would need to accumulate enough independent windows at $T = 10{,}000$ s to achieve a coherence-based error of less than 10% in $\rho_a$. This is the logistical constraint that makes Antarctic MT surveys instrument-season-limited, not data-volume-limited.

**3. Connection to dynamics.** A new MT survey finds that the conductive till layer beneath a fast-moving ice stream has a resistivity of $0.3\,\Omega\cdot\mathrm{m}$ rather than the $1\,\Omega\cdot\mathrm{m}$ assumed in this lab. Using the relationship between till conductivity, porosity, and pore-fluid salinity (Archie's law, [TODO-CITE: archie1942]), sketch the argument for why a more conductive till is likely a softer one, and trace the chain from till softness through basal drag to ice velocity to surface elevation change detectable by {doc}`elevation`. This is the observational thread that connects the present chapter back to {doc}`../thermomechanics/basal-motion` and {doc}`../thermomechanics/hydrology` — and that remains quantitatively open for most Antarctic ice streams.